# DiTFlow on Wan2.1 — Colab

Runtime → Change runtime type → **GPU** (T4 works for the 1.3B; A100 for the 14B).

Colab's disk is temporary: `/content` is wiped when the runtime recycles.
Download or copy to Drive before you lose results.

In [ ]:
!git clone https://github.com/jnheinrich451-eng/ditflow.git
%cd ditflow

In [ ]:
# Already cloned in an earlier session? Just update.
%cd /content/ditflow
!git pull

## Install

No conda, and **do not reinstall torch** — Colab's build is already correct and
replacing it usually breaks the runtime. Only the missing packages are added.

`transformers>=5` breaks diffusers' model imports, so that upper pin matters.

In [ ]:
!pip install -q "diffusers>=0.33" "transformers>=4.44,<5" accelerate ftfy                omegaconf einops imageio imageio-ffmpeg

**Restart the session now** (Runtime → Restart session) — `transformers` almost
certainly changed version. Then continue from the next cell.

In [ ]:
%cd /content/ditflow
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Verify before spending GPU time

Weight-free, CPU, seconds. Checks the port's AMF against this repo's own
`guidance_utils/motion_flow_utils.py` at f64, plus the transformer chain.
If this fails, nothing downstream is trustworthy.

In [ ]:
!python verify_wan_port.py

## Generate

First run downloads Wan2.1-T2V-1.3B from HuggingFace (~14 GB, mostly the
umT5-XXL text encoder). Public weights — no token needed.

The bundled clips are 24 frames, so `num_frames` must be a 4k+1 value ≤ 24;
the config defaults to 21. Add `--low_vram` on a <24 GB GPU, `--verbose` to
watch the guidance loss fall.

In [ ]:
!python motion_guidance_wan.py     --video_path ./assets/bmx-trees.mp4     --prompt "Leopard running up a snowy hill in a forest"     --verbose

## Did guidance actually run?

A suspiciously fast run usually means guidance was skipped. This reports whether
optimised tensors were saved, and echoes the config that produced the output.

In [ ]:
from colab_utils import summarize_run
summarize_run("results_wan")

## Watch it

In [ ]:
from colab_utils import show_run
show_run("results_wan")          # reference beside result

## Get the files off Colab

`zip_results` triggers a browser download and skips `embeds/*.pt` (large, only
needed for `--inject_embeds`). `save_to_drive` survives runtime recycling.

In [ ]:
from colab_utils import zip_results
zip_results("results_wan")

In [ ]:
# Alternative: keep results across runtime restarts.
# from colab_utils import save_to_drive
# save_to_drive("results_wan")

## Sweep

The defaults are transplanted from CogVideoX and **not tuned for Wan**.
`motion_temp` is the highest-leverage knob: Wan applies RMSNorm to q/k, so
attention logits sit on a different scale than the temperature was tuned for.

`--include_baselines` adds backbone and injection-only rows — without them you
can't tell whether guidance is buying motion fidelity or the backbone just looks
good. Resumable, and writes `results.md` after every run.

In [ ]:
!pip install -q opencv-python
!pip install -q git+https://github.com/openai/CLIP.git      # CLIP score
# motion fidelity pulls CoTracker via torch.hub automatically

In [ ]:
!python sweep_wan.py     -v ./assets/bmx-trees.mp4     -p "Leopard running up a snowy hill in a forest"     --motion_temp 1 2 4     --include_baselines     --output_root ./sweeps/first --dry_run     # drop --dry_run to actually run

In [ ]:
from IPython.display import Markdown
Markdown(open("sweeps/first/results.md").read())